# Contrastive Pair Export & Pilot Validation

This notebook joins the prediction log with the cached BridgeData V2 manifest,
runs the pilot validation of the directional-consistency metric against
early-motion ground truth, and exports `pairs.json` for the
external Isaac Sim visualiser.

The model is never loaded here; inputs are the CSVs produced by the previous
notebooks.

Two-frame protocol: predictions are grouped by `(pair_id, frame)`, not
`pair_id` alone, so `referent_selection` pairs (initial frame) and
`placement_relation` pairs (initial and, where a grasp was detected, grasp
frame) are never pooled together. Only grasp-frame `placement_relation` pairs
are analytically meaningful for that category; their initial-frame pairs are
retained only as the expected-null baseline. See
`docs/PROBE_AND_ANALYSIS.md` for the full protocol.

**Prerequisite:** predictions must have been logged with `pair_id`,
`role` (`'a'`/`'b'`), `scene_id`, and `frame` (`'initial'`/`'grasp'`) passed
via `**extra` in `append_prediction_log`, with `scene_id` matching
`episode_index` in the manifest. A log predating `frame` still loads: every
row is treated as `frame='initial'`.


## 1. Mount Drive


In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
PRED_CSV = '/content/drive/MyDrive/openvla_cache/probe_predictions_v3.csv'
CACHE_DIR = '/content/drive/MyDrive/openvla_cache/bridge_multiobj'
OUT_JSON  = '/content/drive/MyDrive/openvla_cache/pairs.json'
MANIFEST_CSV = os.path.join(CACHE_DIR, 'manifest.csv')
print('predictions ->', PRED_CSV)
print('manifest    ->', MANIFEST_CSV)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
predictions -> /content/drive/MyDrive/openvla_cache/probe_predictions_v3.csv
manifest    -> /content/drive/MyDrive/openvla_cache/bridge_multiobj/manifest.csv


## 2. Import `export_pairs.py`

Imports pair assembly helpers from the Drive-synced repository. Mount Drive in
section 1 first. The default path is `/content/drive/Othercomputers/My MacBook Pro/ECS8056`; change
`REPO_DIR` in the next cell if the folder lives elsewhere on Drive.


In [2]:
import sys, os, importlib

# Drive-synced repository (mount Drive first). Change if the folder path differs.
REPO_DIR = '/content/drive/Othercomputers/My MacBook Pro/ECS8056'


def find_repo_dir(anchor):
    """Return the directory holding `anchor` from known Drive paths only."""
    candidates = []
    if REPO_DIR:
        candidates.append(REPO_DIR)
    candidates.extend([
        '/content/drive/Othercomputers/My MacBook Pro/ECS8056',
        '/content/drive/MyDrive/ECS8056',
        '/content/ECS8056',
    ])
    seen = set()
    for d in candidates:
        d = os.path.abspath(d)
        if d in seen:
            continue
        seen.add(d)
        if os.path.isfile(os.path.join(d, anchor)):
            return d
    return None


module_dir = find_repo_dir('export_pairs.py')
if module_dir is None:
    raise FileNotFoundError(
        f"export_pairs.py not found on Google Drive. Mount Drive in section 1, confirm ECS8056 has finished syncing, then set REPO_DIR to the folder that contains export_pairs.py (tried REPO_DIR={REPO_DIR!r}). Also check with:\n  !ls /content/drive/Othercomputers/My MacBook Pro/ECS8056")
if module_dir not in sys.path:
    sys.path.insert(0, module_dir)
sys.modules.pop('export_pairs', None)
importlib.invalidate_caches()

import export_pairs
from export_pairs import load_inputs, build_pairs, write_pairs
print('imported export_pairs.py from', module_dir)
print('BRIDGE_TO_ISAAC =\n', export_pairs.BRIDGE_TO_ISAAC)


imported export_pairs.py from /content/drive/Othercomputers/My MacBook Pro/ECS8056
BRIDGE_TO_ISAAC =
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


## 3. Load the prediction log and manifest

`load_inputs` validates that the prediction log carries the required probe
columns and fails loudly if any are missing.


In [3]:
preds, manifest = load_inputs(PRED_CSV, MANIFEST_CSV)
print(f'{len(preds)} logged predictions | {preds["pair_id"].nunique()} pair ids | '
      f'{preds["scene_id"].nunique()} scenes')
print('predictions per frame:', dict(preds['frame'].value_counts()))
print(f'{len(manifest)} manifest rows')
preds.head(4)

2226 logged predictions | 797 pair ids | 791 scenes
predictions per frame: {'initial': np.int64(1594), 'grasp': np.int64(632)}
2239 manifest rows


,timestamp,instruction,unnorm_key,do_sample,gpu_name,gpu_capability,dtype,seed,torch,transformers,...,a5,a6,scene_id,pair_id,role,frame,spatial_term,category,feasible_both,sample_idx
0,2026-07-26T13:23:22.837563+00:00,Place the can to the left of the pot.,bridge_orig,False,NVIDIA A100-SXM4-80GB,sm_80,bfloat16,42,2.11.0+cu128,4.40.1,...,0.007992,0.996078,0,ep000000_left,a,initial,left,placement_relation,yes,0
1,2026-07-26T13:23:23.444073+00:00,Place the can to the right of the pot.,bridge_orig,False,NVIDIA A100-SXM4-80GB,sm_80,bfloat16,42,2.11.0+cu128,4.40.1,...,-0.009737,0.996078,0,ep000000_left,b,initial,left,placement_relation,yes,0
2,2026-07-26T13:23:24.078268+00:00,Slide the cloth diagonally to the front of the...,bridge_orig,False,NVIDIA A100-SXM4-80GB,sm_80,bfloat16,42,2.11.0+cu128,4.40.1,...,-0.008126,0.996078,2,ep000002_front,a,initial,front,referent_selection,no,0
3,2026-07-26T13:23:24.683984+00:00,Slide the cloth diagonally to the back of the ...,bridge_orig,False,NVIDIA A100-SXM4-80GB,sm_80,bfloat16,42,2.11.0+cu128,4.40.1,...,0.001545,0.996078,2,ep000002_front,b,initial,front,referent_selection,no,0


## 4. Build contrastive pairs

Predictions are grouped by `(pair_id, frame)`; repeated predictions per role
(samples or paraphrases) are averaged, retaining per-axis standard deviation
and count for the effect-size analysis. Groups lacking both roles are
reported and skipped. A `placement_relation` pair_id may appear twice, once
per frame, when both were probed.


In [4]:
pairs, skipped = build_pairs(preds, manifest)
with_gt = sum(1 for p in pairs if 'gt_vector' in p)
print(f'{len(pairs)} complete pairs | {skipped} incomplete (pair_id, frame) groups skipped | '
      f'{with_gt} pairs joined to ground truth')
from collections import Counter
print('pairs per (category, frame):',
      dict(Counter((p.get('category', 'other'), p['frame']) for p in pairs)))

1113 complete pairs | 0 incomplete (pair_id, frame) groups skipped | 1108 pairs joined to ground truth
pairs per (category, frame): {('placement_relation', 'initial'): 409, ('referent_selection', 'initial'): 388, ('placement_relation', 'grasp'): 316}


## 5. Pilot validation - sign agreement against ground truth

Gate for the directional metric and the frame convention: the metrics are
validated against trajectories whose ground-truth motion direction is known.
For each prediction, the sign of the predicted translation on the dominant
ground-truth axis is compared with the ground-truth sign.

Interpretation:
* **Near-zero agreement on one axis** indicates a flipped axis between the
  Bridge action frame and the assumed frame, set the corresponding row of
  `BRIDGE_TO_ISAAC` in `export_pairs.py` and rebuild.
* **Near-chance agreement on all axes** means the check carries no signal on
  this subset; restrict to scenes with large, unambiguous ground-truth motion
  before concluding anything about the frame.
* Model failure is expected on *some* scenes so the gate is systematic, 
  axis-level disagreement, not per-scene misses.


In [5]:
import numpy as np
import pandas as pd

MIN_GT_NORM = 0.01   # exclude near-static ground truth; tune against the data

rows = []
for p in pairs:
    gt = p.get('gt_vector')
    if gt is None:
        continue
    gt_t = np.asarray(gt[:3])
    if np.linalg.norm(gt_t) < MIN_GT_NORM:
        continue
    dom = int(np.argmax(np.abs(gt_t)))
    for role in ('a', 'b'):
        act_t = np.asarray(p[f'action_{role}'][:3])
        rows.append({
            'pair_id': p['pair_id'],
            'frame': p['frame'],
            'role': role,
            'dominant_axis': 'xyz'[dom],
            'gt_sign': float(np.sign(gt_t[dom])),
            'pred_sign': float(np.sign(act_t[dom])),
            'agree': bool(np.sign(act_t[dom]) == np.sign(gt_t[dom])),
        })

pilot = pd.DataFrame(rows)
print(pilot.groupby('dominant_axis')['agree'].agg(['mean', 'count']))
print(f"\noverall sign agreement: {pilot['agree'].mean():.1%} "
      f"over {len(pilot)} predictions ({pilot['pair_id'].nunique()} pairs)")
print('\nsign agreement by frame:')
print(pilot.groupby('frame')['agree'].agg(['mean', 'count']))

                   mean  count
dominant_axis                 
x              0.626736    576
y              0.673237    964
z              0.458333    672

overall sign agreement: 59.6% over 2212 predictions (792 pairs)

sign agreement by frame:
             mean  count
frame                   
grasp    0.337580    628
initial  0.698232   1584


## 5b. Stratified analysis - primary vs. secondary

The evaluation is split to avoid three confounds:

* **Referent vs. placement.** Only `referent_selection` prompts (the spatial
  term selects which object to grasp) probe grounding at the first control step.
  `placement_relation` prompts differ only in a destination, and the first
  action is the reach, identical for both variants, so they are reported
  separately and never pooled into the headline result.
* **Frame.** `placement_relation` pairs may carry both an initial-frame and a
  grasp-frame prediction (see the two-frame protocol). Only the grasp-frame
  predictions are analytically meaningful for that category, since the
  initial-frame reach is identical for both variants by construction; the
  initial-frame subset is reported only as the expected-null baseline and
  never pooled with the grasp-frame result.
* **Feasibility.** Antonym-swapped prompts can imply physically impossible
  placements; the primary stratum keeps only scenes reviewed as feasible on
  **both** sides (`feasible_both == 'yes'`).

**Headline statistic:** the paired difference `action_A − action_B` on the
lateral axis (`dx`, the axis expected to flip for left/right), tested with a
Wilcoxon signed-rank test across pairs. The per-pair sign-flip rate is retained
as a descriptive statistic only.

In [6]:
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

LATERAL_AXIS = 0   # dx: the translation component expected to flip for left/right

# One row per (pair_id, frame), with the lateral-axis paired difference and its
# sign flip.
rec = []
for p in pairs:
    a = float(p['action_a'][LATERAL_AXIS])
    b = float(p['action_b'][LATERAL_AXIS])
    rec.append({
        'pair_id': p['pair_id'],
        'frame': p['frame'],
        'category': p.get('category', 'other'),
        'feasible_both': p.get('feasible_both', 'unreviewed'),
        'spatial_term': p.get('spatial_term', ''),
        'lat_a': a,
        'lat_b': b,
        'diff': a - b,               # action_A - action_B on the lateral axis
        'sign_flip': (a * b) < 0,
    })
df = pd.DataFrame(rec)


def summarise(sub: pd.DataFrame, label: str):
    """Print counts, sign-flip rate (descriptive) and Wilcoxon on the diffs."""
    n = len(sub)
    print(f'\n[{label}] n={n} pairs')
    if n == 0:
        return
    print(f'  sign-flip rate (descriptive): {sub["sign_flip"].mean():.1%}')
    diffs = sub['diff'].to_numpy()
    if n >= 1 and np.any(diffs != 0):
        try:
            stat, pval = wilcoxon(diffs)
            print(f'  Wilcoxon signed-rank (A-B, lateral): '
                  f'W={stat:.1f}, p={pval:.4g}, median diff={np.median(diffs):+.4f}')
        except ValueError as e:
            print(f'  Wilcoxon not computable: {e}')
    else:
        print('  Wilcoxon not computable: all paired differences are zero')


# --- PRIMARY: referent_selection, feasible on both sides (initial frame) ---
primary = df[(df['category'] == 'referent_selection') &
             (df['feasible_both'] == 'yes') &
             (df['frame'] == 'initial')]
print('=' * 60)
print('PRIMARY ANALYSIS')
summarise(primary, 'referent_selection & feasible_both==yes (initial frame)')

# --- SECONDARY strata: reported separately, never pooled into the primary ---
print('\n' + '=' * 60)
print('SECONDARY STRATA (reported separately, not pooled)')
summarise(df[(df['category'] == 'placement_relation') & (df['frame'] == 'grasp')],
          'placement_relation (grasp frame, the decision point)')
summarise(df[(df['category'] == 'placement_relation') & (df['frame'] == 'initial')],
          'placement_relation (initial frame, expected null baseline)')
summarise(df[(df['category'] == 'referent_selection') &
             (df['feasible_both'] != 'yes')],
          'referent_selection & infeasible/unreviewed')
summarise(df[df['category'] == 'other'], 'other')

print('\nstratum sizes:')
print(df.groupby(['category', 'frame', 'feasible_both']).size())

PRIMARY ANALYSIS

[referent_selection & feasible_both==yes (initial frame)] n=7 pairs
  sign-flip rate (descriptive): 0.0%
  Wilcoxon signed-rank (A-B, lateral): W=5.0, p=1, median diff=+0.0000

SECONDARY STRATA (reported separately, not pooled)

[placement_relation (grasp frame, the decision point)] n=316 pairs
  sign-flip rate (descriptive): 24.7%
  Wilcoxon signed-rank (A-B, lateral): W=9606.5, p=0.2634, median diff=+0.0000

[placement_relation (initial frame, expected null baseline)] n=409 pairs
  sign-flip rate (descriptive): 14.9%
  Wilcoxon signed-rank (A-B, lateral): W=9656.0, p=0.00243, median diff=+0.0000

[referent_selection & infeasible/unreviewed] n=381 pairs
  sign-flip rate (descriptive): 23.1%
  Wilcoxon signed-rank (A-B, lateral): W=12496.5, p=0.3753, median diff=+0.0000

[other] n=0 pairs

stratum sizes:
category            frame    feasible_both
placement_relation  grasp    no                 8
                             unreviewed       286
                       

## 6. Export `pairs.json`

Written to Drive, then transferred to the rendering instance, e.g.:

```
scp -i key.pem pairs.json ubuntu@<instance-ip>:~/
```

The visualiser is run on the instance with Isaac Sim's bundled interpreter:
`./python.sh visualise_pairs.py --pairs ~/pairs.json --out ./figs`.


In [7]:
write_pairs(pairs, OUT_JSON)

[write_pairs] wrote 1113 pairs -> /content/drive/MyDrive/openvla_cache/pairs.json


'/content/drive/MyDrive/openvla_cache/pairs.json'

## 7. Preview a pair

Spot check of one exported record: instructions, mean action vectors, and the
x-axis sign relationship that the left/right probes target.


In [8]:
import numpy as np
p = pairs[0]
print('pair_id :', p['pair_id'], '| scene:', p['scene_id'])
print('A:', p['instr_a'])
print('   action =', np.round(p['action_a'], 4), f"(n={p['n_a']})")
print('B:', p['instr_b'])
print('   action =', np.round(p['action_b'], 4), f"(n={p['n_b']})")
dx_a, dx_b = p['action_a'][0], p['action_b'][0]
print(f'dx(A) = {dx_a:+.4f}   dx(B) = {dx_b:+.4f}   '
      f"x-sign flip: {'YES' if dx_a * dx_b < 0 else 'no'}")
if 'gt_vector' in p:
    print('gt      =', np.round(p['gt_vector'], 4))

pair_id : ep000000_left | scene: 0
A: Place the can to the left of the pot.
   action = [-2.700e-03  5.000e-04 -8.000e-03  1.100e-03  2.530e-02  8.000e-03
  9.961e-01] (n=1)
B: Place the can to the right of the pot.
   action = [-2.700e-03  5.000e-04 -8.800e-03  1.100e-03  3.000e-02 -9.700e-03
  9.961e-01] (n=1)
dx(A) = -0.0027   dx(B) = -0.0027   x-sign flip: no
gt      = [-0.0868  0.0915 -0.0258 -0.0594 -0.035   1.4381  1.    ]
